## **1.0 Prompt Engineering for Item Generation**



### **Overview**

This hands-on workshop covers various prompt engineering techniques for generating high-quality items.

Learning Objectives:
- Master different types of prompting strategies
- Implement structured prompt templates for item generation
- Apply few-shot learning for consistent item quality
- Use chain-of-thought reasoning for complex assessments
- Implement evaluation and refinement workflows

### **Basic Concept**
Prompt engineering is the process of crafting effective instructions for language models to get the best possible results. It means carefully choosing words, providing context, and structuring requests in ways that help LLMs produce more accurate, helpful, and on-target responses.

In item development, prompt engineering ensures that generated items are educationally sound, technically accurate, appropriately challenging, and consistently aligned with learning objectives.

#### Prompting Strategies
Effective prompt engineering for item generation employs various strategic approaches, each designed to optimize specific aspects of LLM output.

The working loop looks like this:

```
write a prompt  →  read the output  →  spot what is wrong  →  reword  →  read again  →  …
```

Three strategies do most of the work, and all three predate the current generation of models:

| Strategy | Prompt details |
|---|---|
| **Zero-shot** | Instruction without example. |
| **Few-shot** | Instruction with two to five examples of the desired output. |
| **Chain-of-thought** | Instruction with a request to reason before answering. |


##### **Zero-shot Prompting**
This is the simplest prompting strategy where we give the AI instructions
without any examples. It's like asking someone to do a task by just
explaining what you want, without showing them an example of expected output.

**When to use Zero-Shot:**
1. Quick prototyping and testing
2. Simple, straightforward tasks
3. When you don't have good examples ready
4. For creative tasks where you want variety

**Limitations:**
1. Less consistent formatting
2. May misunderstand complex requirements
3. Output quality can vary

##### **Few-shot Prompting**
Few-shot prompting is like teaching by showing examples. Instead of just
telling the AI what to do, we show it a few examples of inputs and their
desired outputs, and it learns the pattern.

**When to use Few-Shot:**
1. When you need consistent formatting
2. When you have good examples available
3. For complex tasks with specific patterns
4. When zero-shot isn't giving consistent results

**Benefits:**
1. More consistent output format
2. Better understanding of requirements
3. Can encode domain expertise through examples
4. Reduces ambiguity

**Limitations:**
1. Requires good quality examples
2. Uses more tokens (costs more)
3. Can overfit to example patterns

##### **Chain-of-Thought Prompting**
Chain-of-thought prompting guides the AI through explicit step-by-step reasoning processes, making the model's thinking visible and systematic. This approach is particularly valuable for complex assessment item development where multiple considerations must be balanced.

**When to use Chain-of-Thought:**
1. For complex assessment scenarios requiring multi-step analysis
2. When generating items that test higher-order thinking skills
3. For quality assurance and validation of item construction
4. When transparency in the generation process is important
5. For developing items with sophisticated distractors

**Benefits:**
1. Improved reasoning quality: Forces systematic consideration of all aspects
2. Transparency: Makes the generation process auditable and reviewable
3. Better distractor development: Explicit focus on misconception-based options
4. Quality control: Each step can be evaluated independently

**Limitations:**
1. Longer prompts and responses cost more token usage
2. May generate unnecessarily complex items
3. More processing time required
4. Requires careful prompt structure to maintain consistency










The literature says the choice matters. Chan et al. (2025) compared standard prompting against
chain-of-thought with worked examples and chain-of-thought expressed as code, on GPT-3.5 and GPT-4,
and found the chain-of-thought variants produced better items — with the largest gains on getting
the *key* right, which is the failure that matters most for a scored item.

So this is a real technique with measurable effects. We are going to use it properly, and then look
carefully at what it still does not give us.

In [1]:
# Install langchain and langchain google module
%pip install -q -U langchain langchain-google-genai langchain_groq

print("Install finished.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 718.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 9.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.
Install finished.


In [2]:
import os
import re

from google.colab import userdata
#from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

#GEMINI
#GEMINI_MODEL = "gemini-3.5-flash"
#os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# temperature 0.7 is deliberate: it is what a person exploring in a chat window gets.
#llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0.7)

#groq
GROQ_MODEL = "allam-2-7b"
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# temperature 0.7 is deliberate: it is what a person exploring in a chat window gets.
llm = ChatGroq(model=GROQ_MODEL, temperature=0.7)

def diagnose(e):
    """Turn an exception into the most likely cause, in plain language."""
    msg = str(e).lower()
    if "429" in msg or "quota" in msg or "rate limit" in msg:
        return "Free-tier quota or rate limit hit. Wait 60 seconds and re-run this cell."
    if "404" in msg or "not found" in msg:
        return f"Model '{GEMINI_MODEL}' not found. Run the model-list cell in notebook 0.0."
    if "401" in msg or "403" in msg or "api key" in msg:
        return "Key problem. Check GOOGLE_API_KEY in Colab Secrets, with notebook access on."
    return "Unexpected error — read the detail line below."


def ask(prompt, model=llm):
    """Send a plain-text prompt, return plain text. Never raises."""
    try:
        return model.invoke(prompt).content
    except Exception as e:
        return f"[call failed] {diagnose(e)}\n  Details: {type(e).__name__}: {e}"


#print(f"Ready. Model: {GEMINI_MODEL}")
print(f"Ready. Model: {GROQ_MODEL}")

Ready. Model: allam-2-7b


## 1️.0 Zero-shot


In [3]:
ZERO_SHOT = (
    """Write a multiple-choice question for 8th grade science about photosynthesis.
    Give four options and indicate the correct answer."""
)

print(ask(ZERO_SHOT))

Which of the following correctly describes the main purpose of photosynthesis in plants?

A) Converting oxygen into carbon dioxide
B) Converting sunlight into chemical energy
C) Converting water into glucose
D) Converting nitrogen gas into amino acids

Correct answer: B) Converting sunlight into chemical energy 


Read it as an item writer, not as a reader. Typical things to notice:

- The **format** is whatever the model felt like — `A)` or `A.` or a bulleted list; the key at the
  bottom, or in bold, or in a sentence.
- The **distractors** are often wrong in ways no eighth grader is wrong.
- Nothing states **which standard** the item measures, or at what cognitive level.

## 1️.1 Zero-shot (Detailed Instruction)


In [4]:
ZERO_SHOT_long = (
    """You are an experienced item writer creating assessment items for middle school science.
    Write one multiple-choice question for 8th grade students on the topic of photosynthesis.
    The item should align with NGSS MS-LS1-6 (constructing an explanation for the role of
    photosynthesis in the cycling of matter and flow of energy into and out of organisms).

    Requirements for the stem:
    - Write a clear, complete question that can be answered without looking at the options.

    Requirements for the options:
    - Provide exactly four options labeled A, B, C, and D.
    - Include exactly one correct answer that is scientifically accurate and unambiguous.
    - Avoid clues that point to the key, such as repeated words from the stem.

    Output format:
    "Question: <stem>\n"
    "A. <option>\n"
    "B. <option>\n"
    "C. <option>\n"
    "D. <option>\n"
    "Correct Answer: <letter>\n"""
)

print(ask(ZERO_SHOT_long))

Question: How does photosynthesis contribute to the cycling of matter and flow of energy in ecosystems?

A. Photosynthesis helps recycle carbon dioxide from the atmosphere into organic compounds consumed by organisms.
B. Photosynthesis allows plants to convert sunlight into chemical energy, which is essential for their growth.
C. Photosynthesis generates oxygen as a waste product, which is released into the atmosphere and consumed by animals.
D. Photosynthesis links the energy from sunlight to chemical energy stored in food, enabling the flow of energy through food webs.

Correct Answer: D 


## 2️. Few-shot

Show the model what you want instead of describing it. Two well-chosen examples.

In [5]:
FEW_SHOT = """You are an experienced 8th grade science item writer.
Write one multiple-choice item in exactly the style of the examples below.

EXAMPLE 1
Stem: A plant is kept in a dark closet for two weeks. Which change is most likely?

A. The plant gains mass because it stores water.
B. The plant loses mass because it cannot carry out photosynthesis.
C. The plant grows faster because it is not stressed by light.
D. The plant stays the same because roots supply all its food.
Answer: B
Rationale: Without light the plant cannot photosynthesize and consumes stored sugars.

EXAMPLE 2
Stem: Which substance is released into the air during photosynthesis?

A. Nitrogen
B. Carbon dioxide
C. Oxygen
D. Water vapor
Answer: C
Rationale: Oxygen is produced when water molecules are split during the light reactions.

NOW WRITE ONE NEW ITEM
Topic: the role of chlorophyll in capturing light energy
"""

print(ask(FEW_SHOT))

Stem: Chlorophyll plays a crucial role in capturing light energy during the process of photosynthesis. Which of the following statements accurately describes the function of chlorophyll in this process?

A. Chlorophyll absorbs all wavelengths of light, except red, for energy.
B. Chlorophyll reflects light energy instead of absorbing it, causing plants to appear green.
C. Chlorophyll only absorbs blue and red light for energy, while ignoring green light.
D. Chlorophyll is not involved in capturing light energy; this responsibility lies solely with the plant cells.
Answer: A
Rationale: Chlorophyll effectively absorbs light energy, particularly in the blue and red wavelengths, allowing plants to convert light into chemical energy for growth. 


There is usually a visible improvement — the format tracks the examples, and the distractors are
closer to real student thinking. This is the single cheapest upgrade available to most programs,
and it is the reason "just add two examples" is the most useful sentence in prompt engineering.

## 3️. Chain-of-thought

Ask for reasoning before the answer. For item writing, the useful version is not "think step by
step" in the abstract — it is asking for the intermediate object you actually want: the student
misconceptions that the distractors should represent.

In [6]:
CHAIN_OF_THOUGHT = """You are an experienced 8th grade science item writer.

Work in two steps.

Step 1. List three specific misconceptions 8th grade students hold about photosynthesis.
Number them.

Step 2. Write one multiple-choice item on photosynthesis in which each of the three
distractors corresponds to one of the misconceptions from Step 1. Label the stem, the four
options A-D, the answer, and a one-line rationale for each option.
"""

print(ask(CHAIN_OF_THOUGHT))

Step 1: Three specific misconceptions 8th grade students hold about photosynthesis:

1. Misconception 1: Photosynthesis is a process where plants only grow during the daytime.
2. Misconception 2: Photosynthesis only occurs in water-based environments.
3. Misconception 3: Plants can perform photosynthesis without sunlight or carbon dioxide.

Step 2: Multiple-choice item on photosynthesis:

Stem: Photosynthesis is a process through which plants convert sunlight, carbon dioxide, and water into glucose (sugar) and oxygen. However, some misconceptions exist among students. Please consider the following options:

A. Photosynthesis only occurs in water-based environments.
B. Plants can only grow during the night.
C. Photosynthesis requires plants to be underwater.
D. Plants can perform photosynthesis without sunlight or carbon dioxide.

Answer: D. Plants can perform photosynthesis without sunlight or carbon dioxide.

Rationale:

D. Plants need sunlight as a primary source of energy for photos

The distractors now have a *reason to exist*. Each one is a wrong answer some
real student would actually choose, and the rationale tells a reviewer why. That is the difference
between four options and one key plus three pieces of filler.

##

## 4️. Hands-on Activity

In [7]:
ACTIVITY_PROMPT = (
    """You are an experienced ....

    Requirements for the stem:
    - Write a clear, complete question that can be answered without looking at the options.
    -
    -

    Requirements for the options:
    - Provide exactly four options labeled A, B, C, and D.
    -
    -

    Output format:
    "Question: <stem>\n"
    "A. <option>\n"
    "B. <option>\n"
    "C. <option>\n"
    "D. <option>\n"
    "Correct Answer: <letter>\n
    Work in two steps.

    Step 1. List three specific misconceptions ... students hold about ....
    Number them.

    Step 2. Write one multiple-choice item on ... in which each of the three
    distractors corresponds to one of the misconceptions from Step 1. Label the stem, the four
    options A-D, the answer, and a one-line rationale for each option."""

)

print(ask(ACTIVITY_PROMPT))

Step 1: List three specific misconceptions students hold about the process of photosynthesis:
1. Plants only need sunlight to grow.
2. Plants produce oxygen and glucose simultaneously.
3. Carbon dioxide is the limiting factor for plant growth.

Step 2: Write a multiple-choice item about photosynthesis, with the distractors corresponding to the three misconceptions from Step 1:

Question: What is the primary role of photosynthesis in plants?

A. Converting sunlight into chemical energy
B. Producing oxygen and glucose simultaneously
C. Serving as a waste disposal for carbon dioxide
D. Reducing the demand for fossil fuels

Correct Answer: B. Producing oxygen and glucose simultaneously

Rationale:
B. This misconception is related to the misconception number 2 from Step 1, which states that plants produce oxygen and glucose simultaneously during photosynthesis. In reality, photosynthesis produces glucose, not oxygen, and oxygen is a byproduct released into the atmosphere. 


## 5. Generating multiple items

Everything above was a single run. Operationally you will not generate one item; you will generate
hundreds, and the question is whether the thing you tuned holds up across a batch.

#### One item JSON

In [8]:
## genrating with json structure
CHAIN_OF_THOUGHT_multiple = """You are an experienced 8th grade science item writer.

Work in two steps.

Step 1. List three specific misconceptions 8th grade students hold about photosynthesis.
Number them.

Step 2. Write one multiple-choice item on photosynthesis in which each of the three
distractors corresponds to one of the misconceptions from Step 1. Label the stem, the four
options A-D, the answer, and a one-line rationale for each option.

Format your response as a JSON object with this structure:
{{
  "stem": "[question stem]",
  "option_a": "[option]",
  "option_b": "[option]",
  "option_c": "[option]",
  "option_d": "[option]",
  "correct_answer": "[single letter: A, B, C, or D]",
  "rationale_a": "[why option A is correct or incorrect, and which misconception it targets]",
  "rationale_b": "[why option B is correct or incorrect, and which misconception it targets]",
  "rationale_c": "[why option C is correct or incorrect, and which misconception it targets]",
  "rationale_d": "[why option D is correct or incorrect, and which misconception it targets]",
  "misconception1": "[misconception]",
  "misconception2": "[misconception]",
  "misconception3": "[misconception]"
}}

Respond with the JSON object only. Leave out any extra text, explanations, or markdown such as code fences.
"""

print(ask(CHAIN_OF_THOUGHT_multiple))

{
  "stem": "Which of the following statements about photosynthesis is true? ",
  "option_a": "Photosynthesis is the process by which plants use sunlight to produce energy.",
  "option_b": "Plants need sunlight, water, and carbon dioxide to produce oxygen as a byproduct.",
  "option_c": "Photosynthesis does not require sunlight.",
  "option_d": "Plants can perform photosynthesis in the dark.",
  "correct_answer": "B",
  "rationale_a": "The statement accurately explains photosynthesis as the process by which plants use sunlight to produce energy.",
  "rationale_b": "This statement correctly states that plants need sunlight, water, and carbon dioxide to produce oxygen as a result. It addresses misconception #2.",
  "rationale_c": "The statement contradicts the correct understanding of photosynthesis, as it claims photosynthesis does not require sunlight. It addresses misconception #1.",
  "rationale_d": "The statement highlights the byproduct of photosynthesis, oxygen, while also mention

#### Five item JSON

In [10]:
import json
import time
from datetime import datetime

def parse_json(text):
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])

rows = []
n_items = 5
max_tries = 3

for i in range(1, n_items + 1):
    item = None
    for attempt in range(1, max_tries + 1):
        try:
            output = ask(CHAIN_OF_THOUGHT_multiple)
            item = parse_json(output)
            break
        except Exception as e:
            print(f"Item {i}, attempt {attempt}: {type(e).__name__}, retrying...")
            time.sleep(3)

    if item is None:
        print(f"Item {i} failed after {max_tries} tries.")
        continue

    item["correct_answer"] = str(item.get("correct_answer", "")).strip()[:1].upper()
    rows.append({"id": i, **item})
    print(f"Item {i} done")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"cot_items_{timestamp}.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(rows)} of {n_items} items to {filename}")

Item 1 done
Item 2 done
Item 3 done
Item 4 done
Item 5 done

Saved 5 of 5 items to cot_items_20261003_171722.json


#### Read generated JSON output

In [11]:
import pandas as pd

with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df

,id,stem,option_a,option_b,option_c,option_d,correct_answer,rationale_a,rationale_b,rationale_c,rationale_d,misconception1,misconception2,misconception3
0,1,Which statement about photosynthesis is false?...,Photosynthesis occurs only in human lungs.,Plants absorb carbon dioxide and water to prod...,Photosynthesis takes place in leaves at night.,Plants use sunlight to create minerals for ani...,A,The misconception targeted by Option A is that...,The misconception addressed by Option B is tha...,Option C addresses the misconception that phot...,Option D is incorrect because plants use sunli...,Photosynthesis occurs only in human lungs,Photosynthesis occurs at night,Plants use sunlight to create minerals for ani...
1,2,Which of the following statements about photos...,Photosynthesis occurs only in green plants.,Plants use sunlight to turn water into oxygen.,Photosynthesis is the process by which plants ...,Photosynthesis begins when plants take in carb...,D,The first misconception is that photosynthesis...,The second misconception is that plants use su...,The third misconception is that photosynthesis...,The correct statement is that photosynthesis b...,Photosynthesis occurs only in green plants.,NaN,NaN
2,3,Which of the following statements about photos...,Photosynthesis requires sunlight to occur.,Photosynthesis occurs only in plants.,Photosynthesis produces oxygen as a waste prod...,Photosynthesis happens only in animals.,D,"Option A is correct, as sunlight is necessary ...","Option B is incorrect, as photosynthesis also ...","Option C is incorrect, as photosynthesis produ...",NaN,NaN,NaN,NaN
3,4,Which of the following statements about photos...,Photosynthesis occurs only in the human eye.,Photosynthesis is the process by which plants ...,Plants can photosynthesize in water without su...,Photosynthesis is the process by which plants ...,A,The misconception addressed here is that photo...,The misconception here is that plants produce ...,The misconception in this option is that plant...,The correct answer addresses the misconception...,Photosynthesis occurs only in the human eye,Plants produce oxygen as the main outcome,Plants can photosynthesize in water without su...
4,5,Which of the following statements about photos...,Photosynthesis occurs mainly in the roots of p...,Only green plants can perform photosynthesis.,"Photosynthesis requires sunlight, water, and c...",Photosynthesis only happens during the night.,B,Photosynthesis primarily occurs in the chlorop...,Photosynthesis specifically involves green pla...,Photosynthesis is a light-dependent process th...,NaN,Photosynthesis only happens during the night.,NaN,NaN
